# 3. Data Transformation and the Flink API Stack

- Now that we have the "Body" (Architecture) and the "Brain" (Time/Windows) of Flink, we need to look at the "Hands" — the actual tools we use to manipulate data. 

- This section covers how we translate business logic into a series of operators.

## API Layers

- In Flink, at  a high level, there are 2 APIs you can use to define a DAG of operators
    - SQL & Table API: High-level, declarative languages where you don't worry about the underlying implementation
    - DataStream API: The core API for most developers. It offers a balance between control (handling state and windows) and ease of use

- SQL is simplier and less flexible. For the most part, DataStream API is by far the more powerful, and you should use this as far as possible

- To demonstrate these, let's suppose we want to monitor a user session activity. The task is:
    - Clean incoming click logs by casting `event_type` to upper case, and filtering `status` to "LIVE"
    - Attach a 'Plan Type' (Free vs. Pro) from a database 
    - Trigger an alert if a user is inactive for 5 minutes

### SQL / Table API

```sql
-- 1. Define the 'Plan' lookup table (pointing to your DB)
CREATE TABLE user_plans (
    user_id STRING,
    plan_type STRING, -- 'Free' vs 'Pro'
    PRIMARY KEY (user_id) NOT ENFORCED
) WITH (
    'connector' = 'jdbc',
    'url' = 'jdbc:mysql://localhost:3306/user_db',
    'table-name' = 'users'
);

-- 2. The Enrichment Query
SELECT 
    c.user_id, 
    UPPER(c.event_type) AS action, 
    u.plan_type,           -- This is your enrichment!
    c.pro_time
FROM click_logs AS c
LEFT JOIN user_plans FOR SYSTEM_TIME AS OF c.pro_time AS u
    ON c.user_id = u.user_id
WHERE c.status = 'LIVE';
```

- This is the most straightforward approach, because it's writing literal SQL.

- Super easy to clean data and moderately easy to join data, but notice that we completely miss the requirement that we need to handle sessionised alerts!

- Why? Because SQL is not good at handling customised logic. This is the biggest drawback of the SQL API; it may be good for basic processing, but largely unusable for anything custom

### DataStream API

- This will be the meat of your work in Flink. Unless you have a super unique requirement, this is largely sufficient for most of the regular data manipulation tasks you have

- Don't worry too much about the details of each of the operators; we will go through each of them in great detail in the next section.

- For now, focus on the amount of flexibility that the datastream API lets us have compared to SQL

```java

// Filtering status to live
DataStream<Log> liveLogs = rawLogs.filter(log -> log.isLive());

// Set timestamp from ms
DataStream<Log> processedLogs = liveLogs.map(log -> {
    log.setTimestamp(log.getTimestamp() / 1000); 
    return log;
});

// Async I/O to fetch "Free vs Pro" status without blocking the pipeline
DataStream<EnrichedLog> enrichedStream = AsyncDataStream.unorderedWait(
    processedLogs, 
    new DatabaseEnrichmentFunction(), 
    10, TimeUnit.SECONDS, 100
);

// Capture the output of the process function
DataStream<Alert> alerts = enrichedStream
    .keyBy(log -> log.getUserId())
    .process(new SessionTimeoutMonitor());

// Define where those alerts actually go
alerts.addSink(new KafkaSink<>(...)) 
      .name("Alert-Kafka-Sink");

/* +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++ */

// Inside SessionTimeoutMonitor:
public void processElement(EnrichedLog log, Context ctx, Collector<Alert> out) {
    // 1. Update the "Last Seen" state
    lastActivityState.update(log.getTimestamp());
    
    // 2. Register a timer for 5 minutes from now
    long timeout = log.getTimestamp() + (5 * 60 * 1000);
    ctx.timerService().registerEventTimeTimer(timeout);
}

public void onTimer(long timestamp, OnTimerContext ctx, Collector<Alert> out) {
    // 3. If this timer fires, it means no new event arrived to overwrite/postpone it!
    out.collect(new Alert("User " + ctx.getCurrentKey() + " has gone MIA."));
}
```

## DataStream API Details

- Now that we've compared the API layers broadly, let's dive into the actual operator details of the DataStream API

- The DataStream API is built on a few operators, and these range from simple stateless operators to more complex stateful functions

- We will break each of these down with a sample code snippet, to showcase the purpose of each

### Stateless Transformations

- These are just generic operators for cleaning data. They process each element **independently** without knowing about previous events

#### Map

- Transforms one element into **exactly one** other element

```java
// Example: Converting a raw JSON String into a ClickLog object
DataStream<String> rawInput = env.socketTextStream("localhost", 9999);

DataStream<ClickLog> logStream = rawInput.map(new MapFunction<String, ClickLog>() {
    @Override
    public ClickLog map(String value) throws Exception {
        // Parse the JSON string into a POJO
        return ObjectMapper.readValue(value, ClickLog.class);
    }
});

// Example: Using a Lambda to mask a User ID for privacy
DataStream<ClickLog> maskedStream = logStream.map(log -> {
    log.setUserId("MASKED_" + log.getUserId().hashCode());
    return log;
});
```

#### Filter

- The Filter operator keeps elements that meet a specific condition and discards the rest

```java
// Example: Only keeping logs where the status is "LIVE"
DataStream<ClickLog> liveLogs = logStream.filter(new FilterFunction<ClickLog>() {
    @Override
    public boolean filter(ClickLog log) throws Exception {
        return "LIVE".equalsIgnoreCase(log.getStatus());
    }
});

// Example: Using a Lambda to filter out bot traffic
DataStream<ClickLog> humanLogs = liveLogs.filter(log -> !log.getIsBot());
```

#### FlatMap

- FlatMap is used when a single input might result in zero, one, or multiple outputs. It is often used for "unnesting" data.

```java
// Example: A log has a list of 'tags'. We want to emit a separate record for each tag.
DataStream<TagEvent> tagStream = logStream.flatMap(new FlatMapFunction<ClickLog, TagEvent>() {
    @Override
    public void flatMap(ClickLog log, Collector<TagEvent> out) throws Exception {
        // Loop through the list of tags in the log
        for (String tag : log.getTags()) {
            // Emit a new object for every single tag found
            out.collect(new TagEvent(log.getUserId(), tag, log.getTimestamp()));
        }
    }
});
```

### Stateful Transformations

- Stateless transformations are the basic building blocks in flink for data cleaning. But after you clean the data, you hit a problem: what if I want to compare the data I'm seeing now, with the data I saw some time ago?

- This is undoable in stateless operators; they are pure functions and assume data are independent. This is the key difference between stateless and stateful processing!

#### Physical Grouping: `keyBy`

- Before we move on to stateful operators, we need to take a deeper look into stateful processing

- When we say that we want to do "stateful" processing, it comes with the notion of keeping track of some entity over time.
    - This is implied, because without the notion of tracking the same user/purchase/something over time, there is no notion of state
    - That is, state only exists if your data is **NOT** independent

- Since Flink is a distributed system, data from the same unit of analysis (say, some User A), can be spread across many task slots AND many Task Managers!
    - This is because Flink is made up of several task managers, and their corresponding task slots (see section 1)
    - This is a big problem if we want to perform stateful processing for some user
    - If I want to know how many clicks User A performed within a 10 minute period, but all the events of User A are scattered over many different TaskManagers or Task slots, now I need to go around collecting them before processing the count
    - This collection is **extremely** slow

- Thankfully, Flink gives us the `keyBy` method to determine where to send data
    - This is the reason why stateful processing works so well in Flink; it deliberately groups the same unit of data into the same task slot using this `keyBy`
    - This minimises copying (where the task manager has to search around for copies of the same state across different task slots), or network transfer (where different task managers need to serialise and transfer data to each other)
 
- The `keyBy` operator logically partitions a DataStream into a KeyedStream. 
    - Doing this gives us `State` and `Window`

```java
KeyedStream<EnrichedLog, String> keyedStream = enrichedStream
    .keyBy(log -> log.getUserId());
```

#### States and Timers

- Beyond learning about the physical data allocation functionality in `keyBy`, we additionally need to clarify conceptually what we mean by "State"

- Once you perform a `keyBy`, the DataStream turns into a KeyedDataStream, and this gives you granular access to 2 things:
    - **State:** This is used by Flink to "remember" the past
    - **Timer:** This is used by Flink to "anticipate" the future

- These are rarely used in isolation, so they aren't really distinct concepts from a usage POV. But it's important to learn the actual concepts for both of them separately

##### State (aka KeyedState)

- Think of a State (aka KeyedState) as a private "storage locker" for every unique key in your stream (e.g., every user_id)
    - When a record for "User A" arrives, the operator automatically has access to "User A's" specific locker

- There are 5 types of "State" that you can define. Since every "State" is just an intermediate value, the various types of state simply determine the structure of the intermediate that you store

    | Concept | Description | Best For... |
    | :--- | :--- | :--- |
    | ValueState<T> | A single value (e.g., an Integer or an Object). | Storing the "Last seen timestamp" or a "Running total". |
    | ListState<T> | A list of elements. | Buffering events until a specific condition is met (e.g., waiting for "Transaction Complete"). |
    | MapState<K, V> | Key-value pairs stored within the keyed stream. | Keeping track of counts for sub-categories (e.g., User "Alice" clicked "Red" vs "Blue"). |
    | ReducingState<T> | A single value that updates via a ReduceFunction. | Aggregating values automatically as they are added. |
    | AggregatingState<IN, OUT> | Similar to ReducingState, but the input and output types can be different. | Complex running averages or multi-stage calculations. |

- Looking at the code examples below, you will notice that the "State" is intimately connected to 2 methods; `open()` and `processElement()`

- We will cover these in more detail later, but for now, to give a flavour of how "state" is used, let's discuss the 2 methods at a high levele
    - You can think of `open()` as the method called when a parallel sub-task is first initialised. Think of this as the "seed" of any aggregation; these contain the instruction for how a state is "started"
    - `processElement()` is the method called for every record going through your operator

##### State Code examples

- ValueState
    - Used for tracking a single, standalone piece of data for each key
        ```java
        // Declaration
        private transient ValueState<Long> lastSeenState;

        // Usage in open()
        ValueStateDescriptor<Long> descriptor = new ValueStateDescriptor<>("lastSeen", Long.class);
        lastSeenState = getRuntimeContext().getState(descriptor);

        // In processElement()
        Long lastTimestamp = lastSeenState.value();
        lastSeenState.update(ctx.timestamp());
        ```


- ListState
    - Buffering multiple elements that belong to the same key
        ```java
        // Declaration
        private transient ListState<String> eventBuffer;

        // Usage in open()
        ListStateDescriptor<String> descriptor = new ListStateDescriptor<>("events", String.class);
        eventBuffer = getRuntimeContext().getListState(descriptor);

        // In processElement()
        eventBuffer.add(value.getEventId());
        // To iterate: 
        // for (String id : eventBuffer.get()) { ... }
        ```

- MapState
    - Store keyed sub-data to avoid iterating through a whole list to find a specific entry
        ```java
        // Declaration
        private transient MapState<String, Integer> categoryCounts;

        // Usage in open()
        MapStateDescriptor<String, Integer> descriptor = new MapStateDescriptor<>("counts", String.class, Integer.class);
        categoryCounts = getRuntimeContext().getMapState(descriptor);

        // In processElement()
        String category = value.getCategory();
        int currentCount = categoryCounts.contains(category) ? categoryCounts.get(category) : 0;
        categoryCounts.put(category, currentCount + 1);
        ```

- ReducingState
    - Automatically merges new values with the existing state using a ReduceFunction. The input and output types **must be the same**
        ```java
            // Declaration
            private transient ReducingState<Integer> runningSum;

            // Usage in open()
            ReducingStateDescriptor<Integer> descriptor = new ReducingStateDescriptor<>(
                "sum", 
                (v1, v2) -> v1 + v2, // ReduceFunction: logic to combine values
                Integer.class
            );
            runningSum = getRuntimeContext().getReducingState(descriptor);

            // In processElement()
            runningSum.add(value.getAmount()); // Automatically added to the sum
        ```

- AggregatingState
    - The most flexible (and complex) state. It allows the input type, intermediate "accumulator" type, and output type to all be different
        ```java
        // Declaration
        private transient AggregatingState<Integer, Double> averageState;

        // Usage in open()
        AggregatingStateDescriptor<Integer, Tuple2<Integer, Integer>, Double> descriptor = 
            new AggregatingStateDescriptor<>(
                "average",
                new AverageAggregate(), // An AggregateFunction<IN, ACC, OUT>
                TypeInformation.of(new TypeHint<Tuple2<Integer, Integer>>() {})
            );
        averageState = getRuntimeContext().getAggregatingState(descriptor);

        // In processElement()
        averageState.add(value.getScore());
        Double currentAvg = averageState.get();
        ```

##### Timers

- While state is useful, it is not fully sufficient. As we saw above, state is great when we want to trigger an action when we see that something has occurred

- Often though, we want to trigger an action when we see the **absence** of something, not just when data is present. 
    - In such cases, "State" isn't ideal, because you cannot store an "absence" 

- The idea here is that you register a "timer" in the `TimerService`, kind of like setting an alarm that goes off if some condition is met
    - These are **fault-tolerant**, if you streams restart, your timers remember the last state they were in too!

- There are 2 notions of timing here:
    - Event-Time Timers: Based on watermark, do something
    - Processing-Time Timers: Based on TaskManager wall clock, do something

    | Concept | Description | Best For... |
    | :--- | :--- | :--- |
    | **Event-Time Timers** | Fires based on the Watermark (the time embedded in the data). | "If I don't see a 'Checkout' event within 30 minutes of 'Add to Cart', fire an alert." |
    | **Processing-Time Timers** | Fires based on the System Clock of the machine running the code. | "Every 5 minutes of real-world time, clear the cache or heartbeat." |

- As we mentioned above, State and Timers are used together

- But while state is called in `open()` and `processElement()`, timers are usually invoked by an `onTimer()` method

- `open` seeds the operator with the start state of the window, `processElement` does work per unit of data, and `onTimer` controls some trigger when a time is reache

##### Timers Code Examples

- Event-Time Timers
    - Waits for the Watermark to reach a certain point
    - If the data stops flowing, these timers stop "ticking"

        ```java
        public void processElement(Event value, Context ctx, Collector<String> out) throws Exception {
            // Get the timestamp from the record itself
            long eventTime = ctx.timestamp();
            
            // Set a timer for 1 hour after the event occurred
            long timerTimestamp = eventTime + (60 * 60 * 1000);
            
            // Registering using the EventTime service
            ctx.timerService().registerEventTimeTimer(timerTimestamp);
            
            // Store the timestamp in state so we can clean up or validate later
            timerState.update(timerTimestamp);
        }

        @Override
        public void onTimer(long timestamp, OnTimerContext ctx, Collector<String> out) {
            out.collect("Event-time alarm for key " + ctx.getCurrentKey() + " fired at " + timestamp);
        }
        ```

- Processing-Time Timers
    - Fire based on the local system clock of the worker node. They are "wall-clock" timers. They don't care about watermarks or the age of your data

        ```java
        public void processElement(Event value, Context ctx, Collector<String> out) throws Exception {
            // Get the current "real world" time on this machine
            long currentTime = ctx.timerService().currentProcessingTime();
            
            // Set an alarm for exactly 5 minutes from 'right now'
            long timerTimestamp = currentTime + (5 * 60 * 1000);
            
            // Registering using the ProcessingTime service
            ctx.timerService().registerProcessingTimeTimer(timerTimestamp);
        }

        @Override
        public void onTimer(long timestamp, OnTimerContext ctx, Collector<String> out) {
            // This triggers based on the server's clock
            out.collect("Processing-time cleanup triggered for key: " + ctx.getCurrentKey());
        }
        ```

#### Stateful Operators

- Now that we understand how Flink physically groups data (keyBy) and how it remembers information (State), we can look at the operators that actually use these features.

- These operators allow us to perform "rolling" aggregations, where we update some result every time a new event arrives

##### Reduce

- This is the simplest way to aggregate statefully. It takes two elements of the same type and combines them into one

- The input and output types MUST be identical. Examples include Running sum, minimum, maximum
    ```java
    // Example: A rolling sum of purchase amounts per user
    DataStream<Purchase> purchases = ...;

    purchases.keyBy(p -> p.getUserId())
        .reduce(new ReduceFunction<Purchase>() {
            @Override
            public Purchase reduce(Purchase p1, Purchase p2) {
                // Combine two purchases into a "summary" purchase object
                return new Purchase(p1.getUserId(), p1.getAmount() + p2.getAmount());
            }
        });
    ```



##### Aggregate

- More flexible than `reduce`, you allow the intermediate "accumulator" and the final result to be different types from the input

- Examples include: Calculating averages, unique counts, or complex multi-field summaries

    ```java
    // Example: Calculating a running average score
    DataStream<Double> averageScores = scores
        .keyBy(score -> "global-average") // Or key by a specific ID
        .window(TumblingEventTimeWindows.of(Time.minutes(5)))
        .aggregate(new AverageAggregate());

    
    import org.apache.flink.api.common.functions.AggregateFunction;
    // 1. Define the Accumulator to hold the intermediate state
    public class AvgAccumulator {
        public double sum = 0.0;
        public long count = 0L;
    }

    // 2. Implement the AggregateFunction
    // <Double (Input), AvgAccumulator (Intermediate State), Double (Result)>
    public class AverageAggregate implements AggregateFunction<Double, AvgAccumulator, Double> {

        @Override
        public AvgAccumulator createAccumulator() {
            return new AvgAccumulator();
        }

        @Override
        public AvgAccumulator add(Double value, AvgAccumulator acc) {
            acc.sum += value;
            acc.count++;
            return acc;
        }

        @Override
        public Double getResult(AvgAccumulator acc) {
            return acc.count == 0 ? 0.0 : acc.sum / acc.count;
        }

        @Override
        public AvgAccumulator merge(AvgAccumulator a, AvgAccumulator b) {
            a.count += b.count;
            a.sum += b.sum;
            return a;
        }
    }
    ```

##### ProcessFunction/KeyedProcessFunction

- While "reduce" and "aggregate" are kind of guided, the lowest level and most granular access Flink provides is its `processFunction`

- This gives you direct low level access to:
    - State (reading/writing to the "lockers")
    - Timers (setting "alarms" for event-time or processing-time)
    - Side Outputs (sending data to multiple destinations at once)

- `KeyedProcessFunction` vs `ProcessFunction`
    - ProcessFunction works on a datastream before a `KeyBy`, and does not have access to State or Timers
    - Use it for stateless tasks that need more than a `map`; such as routing data to "Side Outputs" based on a condition **without** needing to remember anything about the past

- We will provide an example of both below



- `KeyedProcessFunction`: Use when business logic is too complex for simple SQL or standard aggregations: **detecting the absence of an event**
    - Imagine a payment system where a user "Starts" a transaction. 
    - If they don't "Complete" that transaction within 1 minute, we want to fire a fraud alert or a "Customer Abandoned" notification.

    ```java
    // 1. Define the Stream
    DataStream<TransactionEvent> alerts = transactionStream
        .keyBy(event -> event.getTransactionId())
        .process(new TransactionTimeoutFunction());

    // 2. The ProcessFunction Logic
    public class TransactionTimeoutFunction extends KeyedProcessFunction<String, TransactionEvent, String> {

        // A 'locker' to remember if we saw the start event
        private ValueState<Boolean> isStarted;

        @Override
        public void open(Configuration conf) {
            // Initialize the state when the sub-task starts
            isStarted = getRuntimeContext().getState(new ValueStateDescriptor<>("isStarted", Boolean.class));
        }

        @Override
        public void processElement(TransactionEvent event, Context ctx, Collector<String> out) throws Exception {
            if (event.getType().equals("START")) {
                isStarted.update(true);
                
                // Set a timer for 1 minute (60,000 ms) from now
                long timerTimestamp = ctx.timerService().currentProcessingTime() + 60000;
                ctx.timerService().registerProcessingTimeTimer(timerTimestamp);
                
            } else if (event.getType().equals("COMPLETE")) {
                // If they completed it, clear the state so the timer knows not to alert
                isStarted.clear();
            }
        }

        @Override
        public void onTimer(long timestamp, OnTimerContext ctx, Collector<String> out) throws Exception {
            // If the 'locker' still says true, the COMPLETE event never arrived!
            if (isStarted.value() != null && isStarted.value()) {
                out.collect("ALERT: Transaction " + ctx.getCurrentKey() + " timed out!");
                isStarted.clear();
            }
        }
    }
    ```

- `ProcessFunction`
    - Imagine you have a single stream of Log objects.
    - You want to keep the "LIVE" logs in your main pipeline, but you want to divert "DEBUG" logs and "ERROR" logs to completely separate streams for storage or alerting, without losing any data or needing to remember previous events
    - Since a standard ProcessFunction cannot use State or Timers, we focus on its ability to emit data to Side Outputs
    - You can't use `Map`, because you want to emit 1:N records
    - You can't use a `FlatMap`, because you are emitting different types

        ```java
        // 1. Define OutputTags for side streams
        final OutputTag<Log> debugTag = new OutputTag<Log>("debug-logs"){};
        final OutputTag<Log> errorTag = new OutputTag<Log>("error-logs"){};

        // 2. Apply the ProcessFunction to the main stream
        SingleOutputStreamOperator<Log> mainStream = rawLogs
            .process(new LogRoutingFunction());

        // 3. Extract the side outputs
        DataStream<Log> debugStream = mainStream.getSideOutput(debugTag);
        DataStream<Log> errorStream = mainStream.getSideOutput(errorTag);

        /* ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++ */

        public class LogRoutingFunction extends ProcessFunction<Log, Log> {

            @Override
            public void processElement(Log log, Context ctx, Collector<Log> out) throws Exception {
                
                // 1. Route based on Status
                if ("ERROR".equalsIgnoreCase(log.getStatus())) {
                    // Divert to the Error Side Output
                    ctx.output(errorTag, log);
                    
                } else if ("DEBUG".equalsIgnoreCase(log.getStatus())) {
                    // Divert to the Debug Side Output
                    ctx.output(debugTag, log);
                    
                } else if ("LIVE".equalsIgnoreCase(log.getStatus())) {
                    // Keep in the main stream (standard output)
                    out.collect(log);
                }
                
                // Note: Elements not matching these (e.g., "PENDING") are effectively dropped
            }
        }
        ```

### Rich Functions: Processing with Lifecycle

- We have discussed both stateless and stateful transformations above in detail

- For every operator we discussed, we have a "Rich*" analogue (i.e. `Map` has a `RichMap`, `flatMap` has `RichFlatmap` etc)

- What exactly does this do?


- Imagine you are building a fraud detection system. You need to enrich an incoming stream of transactions with user metadata stored in an external database (like Redis or Cassandra).
    - If you tried to do this with a standard `MapFunction`, every single event would require you to open and close a database connection, which would kill your app (or your DB). 
    - This is where the lifecycle methods of a Rich function save the day

- Rich* operators give you 3 things that manage the lifecycle of the operator:
    - `open()`: Called once during initialization. Use this to set up heavy resources (DB connections, state handles)
    - `close()`: Called once when the task shuts down. Use this for cleanup
    - `RuntimeContext`: Accessible via getRuntimeContext(). Used to create Managed State (ValueState, ListState)

- Let's look at an example of this in action:
    - Let's say we want to monitor machine temperatures
    - If a temperature exceeds a threshold, we want to set a timer to alert us if it doesn't cool down within 1 minute
    - To do this, we use a KeyedProcessFunction

    - We use `open()` to initialize our state. Without this, Flink wouldn't know how to recover our "Last Alert Time" if the job failed
        ```java
        public void open(Configuration parameters) {
            // Accessing RuntimeContext to initialize Managed State
            ValueStateDescriptor<Double> desc = new ValueStateDescriptor<>("last-temp", Double.class);
            lastTemp = getRuntimeContext().getState(desc);
        }
        ```
    
    - Next, we define `processElement()` to handle every incoming event. Here, we check the temperature and decide whether to register a timer.
        ```java
            public void processElement(SensorReading value, Context ctx, Collector<Alert> out) {
                // Check if temp is dangerously high
                if (value.temperature > 100.0) {
                    // Register a timer for 60 seconds from now (Event Time)
                    long timerTimestamp = ctx.timestamp() + 60000;
                    ctx.timerService().registerEventTimeTimer(timerTimestamp);
                }
            }
        ```

    - Finally, we define an `onTimer()`, which lets us trigger the alert once 60 seconds of registered TaskManager time has passed
        ```java
            public void onTimer(long timestamp, OnTimerContext ctx, Collector<Alert> out) {
                // This code runs only when the 60-second window expires
                out.collect(new Alert("Machine " + ctx.getCurrentKey() + " is still overheating!"));
            }
        ```


### MultiStream Operations

- So far, we have operated on single data streams, going from stateless, to stateful, life cycle sensitive operators

- What happens when we need two or more streams to work together?

- In Flink, multi-stream operations are categorized by how they share data and whether they require a common key

- We will explore the following:
    - Low-Level Operator: connect()
        - This is the most flexible way to join two streams 
        - Unlike a standard union, connect() allows the two streams to have different types
    - High-Level Window Joins: join()
        - Similar to SQL joins
    - Convenience method when one stream is much larger than another: broadcast()
        - Same idea as Spark, where you take a small table, and copy it into all your workers to minimise serde cost
    

#### `connect()`

- The most flexible way to join two streams is via the connect() operator. Unlike a standard union, connect() allows the two streams to have different types.

- `KeyedCoProcessFunction`: This is the most common operator used after a connect().

- When you key both streams by the same ID (e.g., order_id), they share the same Keyed State

- To write this, you implement `processElement1()` for the first stream and `processElement2()` for the second. Both methods can read/write to the same state and register shared timers

- Example: Imagine we want to join a stream of orders and a stream of payments together

    ```java
    // Match Orders and Payments
    DataStream<Order> orders = ...;
    DataStream<Payment> payments = ...;

    orders.keyBy(Order::getOrderId)
        .connect(payments.keyBy(Payment::getOrderId))
        .process(new KeyedCoProcessFunction<String, Order, Payment, String>() {
            // Shared state for matching
            private ValueState<Order> orderState;
            
            @Override
            public void processElement1(Order o, Context ctx, Collector<String> out) {
                // Handle Order: Check if Payment is already in state or buffer this Order
            }

            @Override
            public void processElement2(Payment p, Context ctx, Collector<String> out) {
                // Handle Payment: Check if Order is already in state
            }
        });
    ```

#### `join()`

- If you need to join streams based specifically on time proximity, Flink provides high-level join operators that handle the state management for you
    - Window Join: Joins elements that fall into the same discrete time window (e.g., Tumbling or Sliding)
    - Interval Join: Specifically designed for "Event Time" streams. It joins Stream A with Stream B if B’s timestamp is within a relative range of A (e.g., A.timestamp - 5min < B.timestamp < A.timestamp + 1min)

- Window Join: A Window Join requires both streams to fall into the same "bucket" of time to be joined

    ```java
    // Join Orders and Shipments that occur in the same 10-minute window
    DataStream<String> windowJoined = orders
        .join(shipments)
        .where(Order::getOrderId)     // Key for the first stream
        .equalTo(Shipment::getOrderId) // Key for the second stream
        .window(TumblingEventTimeWindows.of(Time.minutes(10)))
        .apply(new JoinFunction<Order, Shipment, String>() {
            @Override
            public String join(Order first, Shipment second) {
                return "Order " + first.getOrderId() + " shipped at " + second.getTimestamp();
            }
        });
    ```

- Interval Join: The Interval Join is often preferred in streaming because it joins elements based on a relative time offset rather than fixed window boundaries
    ```java
    // Join Clicks with Conversions that happen within 1 hour AFTER the click
    // Logic: click.timestamp <= conversion.timestamp <= click.timestamp + 1 hour
    DataStream<String> intervalJoined = clickStream
        .keyBy(Click::getUserId)
        .intervalJoin(conversionStream.keyBy(Conversion::getUserId))
        .between(Time.minutes(0), Time.hours(1)) // Lower and upper bound
        .process(new ProcessJoinFunction<Click, Conversion, String>() {
            @Override
            public void processElement(Click click, Conversion conv, Context ctx, Collector<String> out) {
                out.collect("User " + click.getUserId() + " converted after click: " + click.getAdId());
            }
        });
    ```

#### `broadcast()`

- Sometimes you don't want to join two massive streams by a key. Instead, you want to "tune" a high-volume stream using a small stream of rules or configurations.
    - Mechanism: The "Rule" stream is broadcast to every parallel sub-task of the main data operator.
    - Broadcast State: Each sub-task keeps a local copy of the rules in a specialized MapState.
    - Benefit: You can change business logic (e.g., updating a fraud threshold) in real-time by sending a new rule event, without restarting the job


    ```java
    // 1. Define the state descriptor for the broadcasted rules
    MapStateDescriptor<String, Rule> ruleStateDescriptor = new MapStateDescriptor<>(
        "RulesConfig", 
        String.class, 
        Rule.class
    );

    // 2. Broadcast the rule stream
    BroadcastStream<Rule> broadcastRules = ruleStream.broadcast(ruleStateDescriptor);

    // 3. Connect the main data stream with the broadcasted rules
    DataStream<Alert> alerts = userActionStream
        .connect(broadcastRules)
        .process(new DynamicRuleEvaluator());

    /* +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++ */

    public class DynamicRuleEvaluator extends BroadcastProcessFunction<UserAction, Rule, Alert> {

        // Descriptor is needed to access the state in both methods
        private final MapStateDescriptor<String, Rule> ruleStateDescriptor = new MapStateDescriptor<>(
            "RulesConfig", String.class, Rule.class);

        @Override
        public void processElement(UserAction action, ReadOnlyContext ctx, Collector<Alert> out) throws Exception {
            // Read-only access to the broadcast state
            ReadOnlyBroadcastState<String, Rule> rules = ctx.getBroadcastState(ruleStateDescriptor);
            
            Rule activeRule = rules.get(action.getCountry());
            if (activeRule != null && activeRule.getEventType().equals(action.getType())) {
                out.collect(new Alert("Rule Triggered for: " + action.getUserId()));
            }
        }

        @Override
        public void processBroadcastElement(Rule newRule, Context ctx, Collector<Alert> out) throws Exception {
            // Writable access to update the rules for ALL sub-tasks
            BroadcastState<String, Rule> state = ctx.getBroadcastState(ruleStateDescriptor);
            state.put(newRule.getCountry(), newRule);
        }
    }
```

## Conclusion

- We have covered the full spectrum of operators in Flink, from the most basic stateless methods, to handling of state, to multistream with states. 

- We'll end by looking at the Connectors and Serialization in Flink